# 01: PySpark Structured Streaming Fundamentals
> **Data Engineering Diaries** | Real-Time Stream Processing with PySpark & Delta Lake

---

## Overview
Welcome to **Structured Streaming in PySpark**. Structured Streaming is a fast, scalable, fault-tolerant stream processing engine built on Spark SQL.

In this notebook, you will learn:
1. **Streaming SparkSession Setup** with Kryo serialization & Delta Lake configurations.
2. **File Streaming (Memory & Directory Streams)** using `readStream`.
3. **Stateless Stream Transformations** (filtering, parsing, calculated columns).
4. **Watermarking & Windowed Aggregations** (tumbling and sliding event-time windows).
5. **Deduplication** for exactly-once processing guarantees.
6. **Streaming Sinks & Output Modes** (`append`, `update`, `complete`, `memory`, `console`).


In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# Initialize test SparkSession
spark = (
    SparkSession.builder
    .appName("notebook-streaming-fundamentals")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.conf.get('spark.master')}")


## 1. Streaming Rate Source (Synthetic Data Generator)
Spark includes a built-in `rate` streaming source that continuously generates rows with a timestamp and incrementing ID.
This allows testing streaming queries without external Kafka or cloud dependencies.


In [ ]:
# Create streaming rate source generating 5 rows/sec
rate_stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

print(f"Is DataFrame Streaming? {rate_stream_df.isStreaming}")
rate_stream_df.printSchema()


## 2. Watermarking and Tumbling Window Aggregation
A **Watermark** tracks event time progression and dictates when state entries can be safely pruned.
Below we define a 10-second tumbling window aggregation over generated timestamps.


In [ ]:
# Apply 10-second watermark and aggregate in 10-second tumbling windows
windowed_df = (
    rate_stream_df
    .withWatermark("timestamp", "10 seconds")
    .groupBy(F.window(F.col("timestamp"), "10 seconds"))
    .agg(
        F.count("*").alias("generated_count"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value")
    )
)

# Output stream to memory sink for interactive querying in notebook
query = (
    windowed_df.writeStream
    .format("memory")
    .queryName("rate_window_counts")
    .outputMode("complete")
    .start()
)

query.awaitTermination(timeout=6)
query.stop()

# Query memory table
spark.sql("SELECT * FROM rate_window_counts ORDER BY window.start DESC").show(truncate=False)
